# 02 — Frozen patient split and shared snapshot preprocessing

All 49 predictors come from already encoded V63 `MODEL_DATA` on the exact `(PATIENT_ID, END_DT)` key. The same snapshot vector is repeated only at enrollment-valid positions; all other positions are zero and masked. No historical feature values are reconstructed and no monthly feature values are aggregated into quarterly values.

MONTHLY is `[N, 12, 49]`; QUARTERLY is `[N, 4, 49]`. The newest position is index 0. A quarterly position is valid exactly when any of its three corresponding monthly positions is valid. Any inclusive enrollment-interval overlap is sufficient, including one day; this does not assert full-period or separate MX/RX coverage.

Run notebook 01 first, review its source/coverage audits, and copy the printed `EXPECTED_PREPARED_MANIFEST_SHA256` into 02–04. Use the fresh identifiers `D20260927S` / `R20260927S` and output namespace `TAK861_TX_READY_V63_DL_POC_SNAPSHOT_PAD_V1`. There are no dependencies on old experimental tensors, models or run reports. The frozen snapshot and patient-split tables remain the requested source population.

Imputation/scaling is fitted once per eligible TRAIN snapshot, then the identical state is used for both representations. Thus more covered periods do not increase a snapshot's preprocessing weight. This is still snapshot weighting, not one-record-per-patient training. All-padded snapshots remain in model training and reported populations, but contribute no unavailable feature values to the preprocessing fit.

The comparison measures sensitivity to sequence length, positional parameters and enrollment masks on repeated snapshot values. It does not establish temporal feature learning, a quarterly aggregation advantage, or the causal benefit of padding. Upstream V63 encoding used RESP and its original fit population is not verified against the frozen held-out split; results remain retrospective. Keep credentials and row-level artifacts private and clear outputs before sharing.


### Configuration and experiment identity

**Why:** Select the fresh snapshot-padding experiment (`D20260927S` / `R20260927S`, namespace `SNAPSHOT_PAD_V1`) and its exact prepared-input fingerprint. Set the manifest hash printed by notebook 01 before running dependent cells; keep identifiers and settings consistent across notebooks.

**Expected output:** This cell defines configuration and normally displays nothing. Missing secure connection settings or invalid identifiers stop execution.

**Interpretation:** MONTHLY and QUARTERLY contain the same 49 encoded snapshot features repeated over 12 or 4 eligible positions; they are different enrollment-mask views, not reconstructed histories or quarterly feature aggregates. L1/L2 and network settings control training, not a manually assigned feature priority.

In [ ]:
# Use the same fresh identifiers and reviewed manifest hash in 01–04.
import os
import re
os.environ.setdefault('CUBLAS_WORKSPACE_CONFIG', ':4096:8')
if 'sf_options' not in globals() or not isinstance(sf_options, dict) or 'spark' not in globals():
    raise RuntimeError('Supply sf_options securely on the approved Spark runtime; never paste credentials into this notebook.')
DATABASE = 'DSVC_TAKEDA_TA_PRIVATE'
sf_options_dl_poc = dict(sf_options)
sf_options_dl_poc.update(sfDatabase=DATABASE, sfSchema='DS_ML')
SOURCE_PREFIX = 'TAK861_TX_READY_V63'
PREFIX = SOURCE_PREFIX + '_DL_POC'
DATASET_ID = 'D20260927S'
RUN_ID = 'R20260927S'
EXPECTED_PREPARED_MANIFEST_SHA256 = None  # Set only after reviewing the clean notebook-01 export.
if any(not re.fullmatch(r'[A-Z][A-Z0-9_]{0,15}', v) for v in (DATASET_ID, RUN_ID)):
    raise ValueError('Use matching short uppercase dataset/run identifiers in all notebooks.')
EXPERIMENT_PREFIX = PREFIX + '_SNAPSHOT_PAD_V1'
PREPARED_TABLE = EXPERIMENT_PREFIX + '_' + DATASET_ID + '_INPUTS'
SPLIT_TABLE = EXPERIMENT_PREFIX + '_' + DATASET_ID + '_SPLIT'
RUN_PREFIX = EXPERIMENT_PREFIX + '_' + DATASET_ID + '_' + RUN_ID
PREPARED_NAMES = {'population.json', 'manifest.json', 'MONTHLY.npz', 'QUARTERLY.npz'}
SPLIT_NAMES = {'split.json', 'preprocessing.json', 'audit.json'}
MODEL_NAMES = {'checkpoint.pt', 'summary.json', 'history.json'}
MODEL_SETTINGS = dict(d_model=128, n_heads=4, encoder_layers=2, feedforward_dim=256, dropout=.2)
TRAINING_SETTINGS = dict(seed=42, epochs=20, patience=5, min_delta=1e-4, batch_size=64,
                        learning_rate=.001, weight_decay=0., grad_clip=1., device='auto',
                        l1_lambda=1e-6, l2_lambda=1e-4,
                        regularization_scope='linear_and_attention_weights')
CONSUMER_IMPLEMENTATION_SHA256 = 'a06e90d8e6ee4b87b6cba1503ee7ed3c3b580db9c4d8b7c2c87c60ea7d8cf4cb'


### Runtime libraries and artifact layout

**Why:** Load the libraries needed by subsequent cells and declare the columns used to store verified artifacts in the private warehouse. Imports define capabilities; they do not query patient data or train a model.

**Expected output:** Normally none. An import failure means the approved runtime needs that dependency before continuing.

**Interpretation:** A successful import only confirms library availability. It does not validate sources, tensors, feature rankings, or model performance. Any mask convention declared here uses `valid=True` for an enrollment-eligible position; eligibility does not establish that the repeated feature value was observed historically in that period.

In [ ]:
import io
import json
import hashlib
import numpy as np
import pandas as pd
import base64
ARTIFACT_COLUMNS = ["ARTIFACT_NAME", "CHUNK_INDEX", "CHUNK_COUNT", "BYTE_LENGTH", "SHA256", "PAYLOAD_BASE64"]
import math


### Verified artifact read/write helpers

**Why:** Define source reads and chunked artifact storage with length and SHA-256 checks. These checks connect downstream calculations to the exact inputs produced by notebook 01 and prevent silently replacing a different result.

**Expected output:** None while these functions are defined. Later saves report successful write/read-back verification or stop on a mismatch.

**Interpretation:** Matching hashes establish artifact integrity, not clinical correctness or absence of modeling bias. An existing destination with different content is an experiment identity conflict; use a deliberately new identifier after reviewing the change. This rebuild never needs a checkpoint from a previous experiment.

In [ ]:
def require(condition, message):
    if not condition:
        raise ValueError(message)


def array_hash(values):
    values = np.asarray(values)
    h = hashlib.sha256(canonical_json(list(values.shape)).encode())
    h.update(np.isnan(values).astype('u1').tobytes())
    h.update(np.nan_to_num(values, nan=0).astype('<f8').tobytes())
    return h.hexdigest()


def canonical_json(value):
    return json.dumps(value, sort_keys=True, ensure_ascii=False, separators=(",", ":"), allow_nan=False)


def digest_json(value):
    return hashlib.sha256(canonical_json(value).encode()).hexdigest()


def read_table(table):
    return (spark.read.format("snowflake").options(**sf_options_dl_poc)
            .option("dbtable", table).load())


def table_exists(table):
    if not re.fullmatch(r"[A-Z][A-Z0-9_]*", table):
        raise ValueError("Use uppercase letters, numbers and underscores in table names.")
    query = ("SELECT TABLE_NAME FROM DSVC_TAKEDA_TA_PRIVATE.INFORMATION_SCHEMA.TABLES "
             f"WHERE TABLE_SCHEMA = 'DS_ML' AND TABLE_NAME = '{table}'")
    return (spark.read.format("snowflake").options(**sf_options_dl_poc)
            .option("query", query).load().limit(1).count() > 0)


def pack_artifacts(artifacts, chunk_size=50000):
    rows = []
    for name, blob in artifacts.items():
        encoded = base64.b64encode(blob).decode("ascii")
        pieces = [encoded[i:i + chunk_size] for i in range(0, len(encoded), chunk_size)] or [""]
        digest = hashlib.sha256(blob).hexdigest()
        rows.extend((name, i, len(pieces), len(blob), digest, piece)
                    for i, piece in enumerate(pieces))
    return rows


def unpack_artifacts(rows, expected_names):
    groups = {}
    for row in rows:
        name, i, count, size, digest, payload = tuple(row)
        if any(value != int(value) for value in (i, count, size)):
            raise ValueError("Nonintegral artifact chunk metadata.")
        groups.setdefault(name, []).append((int(i), int(count), int(size), digest, payload))
    if set(groups) != set(expected_names):
        raise ValueError("Missing or unexpected saved artifacts.")
    result = {}
    for name, pieces in groups.items():
        pieces.sort(key=lambda p: p[0])
        count, size, digest = pieces[0][1:4]
        if (count < 1 or size < 0 or len(pieces) != count
                or [p[0] for p in pieces] != list(range(count))
                or any(p[1:4] != (count, size, digest) for p in pieces)):
            raise ValueError("Missing, duplicate or inconsistent artifact chunks.")
        blob = base64.b64decode("".join(p[4] for p in pieces), validate=True)
        if len(blob) != size or hashlib.sha256(blob).hexdigest() != digest:
            raise ValueError("Artifact length/hash mismatch.")
        result[name] = blob
    return result


def read_artifacts(table, expected_names):
    rows = (spark.read.format("snowflake").options(**sf_options_dl_poc)
            .option("dbtable", table).load().select(*ARTIFACT_COLUMNS).collect())
    return unpack_artifacts(rows, expected_names)


def save_artifacts(table, artifacts):
    # A matching existing result can be verified after an interrupted read-back.
    if table_exists(table):
        if read_artifacts(table, artifacts) != artifacts:
            raise FileExistsError("Destination contains different artifacts; choose a new RUN_ID.")
        print(f"Existing artifacts verified: DSVC_TAKEDA_TA_PRIVATE.DS_ML.{table}")
        return
    schema = ("ARTIFACT_NAME STRING, CHUNK_INDEX INT, CHUNK_COUNT INT, "
              "BYTE_LENGTH LONG, SHA256 STRING, PAYLOAD_BASE64 STRING")
    frame = spark.createDataFrame(pack_artifacts(artifacts), schema=schema)
    (frame.write.format("snowflake").options(**sf_options_dl_poc)
     .option("dbtable", table).option("truncate_columns", "off")
     .mode("errorifexists").save())
    if read_artifacts(table, artifacts) != artifacts:
        raise ValueError("Saved artifact read-back differs from the completed run.")
    print(f"Saved and verified: DSVC_TAKEDA_TA_PRIVATE.DS_ML.{table}")


### Authoritative features and frozen V63 encoding

**Why:** Require all 49 feature names, in the configured order, to agree between `MODEL_TYPE.FEATURES`, `FINAL_MODEL.SEQ`, and the preparation manifest. Validate each frozen encoding parameter and the source audit.

**Expected output:** None on definition. A later mismatch displays the conflicting contract and raises an error.

**Interpretation:** All features in this experiment must be `V63_SNAPSHOT`, joined to the exact patient/date and broadcast only to eligible positions. Existing `MODEL_DATA` values are already encoded; do not apply the business transformation again. Frozen upstream encoding was not verified as fitted on TRAIN alone, so this remains a retrospective limitation. Feature order and encoding caps do not rank feature importance.

In [ ]:
def parse_features(value):
    if isinstance(value, str):
        value = json.loads(value)
    if isinstance(value, np.ndarray):
        value = value.tolist()
    if not isinstance(value, list) or not value or any(not isinstance(x, str) or not x.strip() or x != x.strip() for x in value):
        raise ValueError("FEATURES must be a nonempty array of exact column names.")
    if len(set(x.upper() for x in value)) != len(value):
        raise ValueError("Duplicate feature names.")
    prohibited = {"PATIENT_ID", "START_DT", "END_DT", "RESP", "SPLIT", "RND", "SCORE", "DECILE", "CENTILE", "MILLILE"}
    if prohibited.intersection(x.upper() for x in value):
        raise ValueError("The selected list contains an identifier, target, split, random helper or prediction output.")
    return value


def _v63_require(condition, message):
    if not condition:
        raise ValueError(message)


def _v63_features(features):
    _v63_require(isinstance(features, (list, tuple, np.ndarray)),
                 'Authoritative features must be an ordered sequence.')
    result = list(features)
    _v63_require(0 < len(result) <= 49, 'Retained predictors must be a nonempty subset of the 49 candidates.')
    _v63_require(all(isinstance(name, str) and name and name == name.strip()
                     for name in result), 'Feature names must be exact, nonempty strings.')
    _v63_require(len({name.upper() for name in result}) == len(result),
                 'Duplicate authoritative features are not permitted.')
    prohibited = {'PATIENT_ID', 'END_DT', 'START_DT', 'RESP', 'SPLIT',
                  'SCORE', 'DECILE', 'CENTILE', 'MILLILE', 'RND'}
    _v63_require(not prohibited.intersection(name.upper() for name in result),
                 'Identifiers, labels and prediction outputs cannot be predictors.')
    return result


def _v63_cap(value, feature):
    _v63_require(not isinstance(value, (bool, np.bool_)),
                 feature + ': VALUE_P must be numeric, not boolean.')
    _v63_require(not isinstance(value, (complex, np.complexfloating)),
                 feature + ': VALUE_P must be real.')
    try:
        cap = float(value)
    except (TypeError, ValueError, OverflowError) as error:
        raise ValueError(feature + ': invalid VALUE_P.') from error
    _v63_require(math.isfinite(cap) and cap >= 0,
                 feature + ': VALUE_P must be finite and nonnegative.')
    return cap


def _v63_type(value, feature):
    _v63_require(isinstance(value, str), feature + ': VAR_TYP must be a declared type.')
    normalized = value.strip().upper()
    _v63_require(normalized in {'BINARY', 'NUMERIC'},
                 feature + ': VAR_TYP must be Binary or Numeric; Dropped/unknown types cannot be silently removed.')
    return {'BINARY': 'Binary', 'NUMERIC': 'Numeric'}[normalized]


def _v63_parameter_hash(records):
    encoded = json.dumps(records, sort_keys=True, ensure_ascii=False,
                         separators=(',', ':'), allow_nan=False).encode('utf-8')
    return hashlib.sha256(encoded).hexdigest()


def fixed_v63_parameters(features, summary):
    """Return (selected ordered parameter records, SHA256) from frozen summary rows.

    FEATURES_SUMMARY may contain additional rows; they do not alter the explicit
    selected feature list. Every summary feature name must be unique, and every
    authoritative feature must have exactly one valid parameter row. No row is
    chosen by importance, VALUE_P magnitude, rank, type, or observed input values.
    """
    features = _v63_features(features)
    _v63_require(isinstance(summary, pd.DataFrame), 'FEATURES_SUMMARY must be a DataFrame.')
    required = ['FEATURES', 'VALUE_P', 'VAR_TYP']
    _v63_require(set(required).issubset(summary.columns),
                 'FEATURES_SUMMARY must provide FEATURES, VALUE_P and VAR_TYP.')
    _v63_require(not summary.columns.duplicated().any(), 'Duplicate summary columns are invalid.')
    names = summary['FEATURES'].tolist()
    _v63_require(all(isinstance(name, str) and name and name == name.strip() for name in names),
                 'Summary feature names must be exact, nonempty strings.')
    _v63_require(len({name.upper() for name in names}) == len(names),
                 'Duplicate FEATURES_SUMMARY feature rows are not permitted.')
    indexed = summary.set_index('FEATURES', verify_integrity=True)
    missing = [name for name in features if name not in indexed.index]
    _v63_require(not missing, 'Missing fixed parameter rows: ' + ', '.join(missing))
    records = []
    for order, name in enumerate(features):
        row = indexed.loc[name]
        cap = _v63_cap(row['VALUE_P'], name)
        kind = _v63_type(row['VAR_TYP'], name)
        transform = ('AGE_CEIL_DECADE' if name.upper() == 'AGE'
                     else 'BINARY_POSITIVE' if kind == 'Binary'
                     else 'NUMERIC_FIXED_CAP')
        records.append({'FEATURE_ORDER': order, 'FEATURES': name, 'VALUE_P': cap,
                        'VAR_TYP': kind, 'TRANSFORM': transform})
    return records, _v63_parameter_hash(records)


def configured_features():
    config = read_table(SOURCE_PREFIX + '_MODEL_TYPE').select('MODEL_TYPE', 'FEATURES').collect()
    require(len(config) == 1, 'Expected one frozen V63 MODEL_TYPE configuration.')
    features = parse_features(config[0]['FEATURES'])
    require(len(features) == 49, 'Exactly 49 V63 model predictors are required.')
    final = read_table(SOURCE_PREFIX + '_FINAL_MODEL').select('FEATURES', 'SEQ').toPandas()
    require(not final[['FEATURES', 'SEQ']].isna().any().any(), 'FINAL_MODEL has missing names/order.')
    require(not final.FEATURES.duplicated().any() and not final.SEQ.duplicated().any(), 'FINAL_MODEL has duplicate feature/order rows.')
    final['SEQ'] = pd.to_numeric(final.SEQ, errors='raise')
    require(np.isfinite(final.SEQ).all() and final.SEQ.eq(np.floor(final.SEQ)).all(), 'FINAL_MODEL SEQ must be finite integer ranks.')
    final_features = final.sort_values('SEQ').FEATURES.tolist()
    comparison = {'configured_model_type': str(config[0]['MODEL_TYPE']), 'configured_feature_count': len(features),
        'final_model_feature_count': len(final_features), 'configured_not_in_final_model': sorted(set(features) - set(final_features)),
        'final_model_not_in_configuration': sorted(set(final_features) - set(features)),
        'final_model_order_matches': features == final_features,
        'ordering_rule': 'MODEL_TYPE.FEATURES retained; FINAL_MODEL ORDER BY SEQ must agree before constructing tensors',
        'feature_role': 'Frozen candidate contract; retained subset declared by preparation manifest'}
    if features != final_features:
        display(pd.DataFrame([comparison]))
        display(pd.DataFrame({'MODEL_TYPE_ORDER': pd.Series(features), 'FINAL_MODEL_SEQ_ORDER': pd.Series(final_features)}))
        raise ValueError('V63 feature lists/order disagree. Resolve source discrepancy; do not silently reorder or select features.')
    return features, comparison


def load_business_parameters(features):
    summary = read_table(SOURCE_PREFIX + '_FEATURES_SUMMARY').toPandas()
    parameters, parameter_hash = fixed_v63_parameters(features, summary)
    contract = {'version': 1, 'parameters': parameters, 'parameter_sha256': parameter_hash,
        'source': DATABASE + '.DS_ML.' + SOURCE_PREFIX + '_FEATURES_SUMMARY',
        'input_space': 'already encoded V63 MODEL_DATA snapshot values', 'encoding_applied_here': False, 'output_space': 'V63 MODEL_DATA business encoding',
        'snapshot_MODEL_DATA_already_encoded': True, 'caps_or_types_refitted_here': False,
        'upstream_fitting_uses_RESP': True,
        'upstream_fit_population': 'Not verified against frozen TRAIN/VALIDATION/TEST; fixed parameter reuse is retrospective',
        'formula': 'AGE ceil(raw/10); Binary raw>0; Numeric raw>VALUE_P => 1 else raw/(VALUE_P+1)'}
    return parameters, summary, contract


def selected_features(manifest):
    candidates, _ = configured_features()
    require(manifest.get('source_features') == candidates, 'Frozen candidate names/order changed.')
    features = parse_features(manifest.get('features'))
    require(features == candidates, 'Snapshot-only input must retain all frozen candidates in their original order.')
    exclusions = manifest.get('excluded_features')
    require(isinstance(exclusions, list) and all(isinstance(r, dict) for r in exclusions),
            'An explicit exclusion audit is required, including an empty list when none are excluded.')
    require([r.get('FEATURE_NAME') for r in exclusions] == [f for f in candidates if f not in features],
            'Every excluded candidate must be accounted for once, in original order.')
    require(all(isinstance(r.get('REASON'), str) and r['REASON'].strip() for r in exclusions),
            'Each excluded feature needs a specific reason.')
    audit = manifest.get('audit')
    require(isinstance(audit, list) and all(isinstance(r, dict) for r in audit), 'Missing retained-feature audit.')
    require([r.get('FEATURE_NAME') for r in audit] == features, 'Retained feature audit is misordered.')
    for row in audit:
        require(row.get('STATUS') == 'IMPLEMENTED_VALIDATED',
                'Each feature extraction path must be implemented and validated.')
        require(row.get('VALUE_SOURCE') == 'V63_SNAPSHOT', 'Every feature must declare V63_SNAPSHOT in this snapshot-only experiment.')
        for key in ('SOURCE', 'CALCULATION', 'LOOKBACK', 'COVERAGE_RULE', 'TEMPORAL_JUSTIFICATION', 'VALIDATION'):
            require(isinstance(row.get(key), str) and bool(row[key].strip()),
                    row['FEATURE_NAME'] + ': missing audit field ' + key)
        if row['VALUE_SOURCE'] == 'V63_SNAPSHOT':
            require(row.get('SOURCE') == DATABASE + '.DS_ML.' + SOURCE_PREFIX + '_MODEL_DATA',
                    'Snapshot fallback must use the matching V63 MODEL_DATA source.')
            require(row.get('SNAPSHOT_JOIN') == ['PATIENT_ID', 'END_DT']
                    and row.get('ALREADY_ENCODED') is True
                    and row.get('BROADCAST') == 'valid_timesteps_only',
                    'Snapshot fallback must match the exact snapshot, stay encoded and respect padding.')
            require(isinstance(row.get('FALLBACK_REASON'), str) and bool(row['FALLBACK_REASON'].strip()),
                    'Snapshot fallback requires a reason.')
    return features


### Prepared inputs, cohort, and split validation helpers

**Why:** Define checks for the reviewed manifest, tensor shapes, zero padding, source hashes, exact snapshot keys/labels, and frozen patient assignments. The snapshot values must be constant across every eligible position and agree between representations.

**Expected output:** No result table until these helpers are called. Invalid inputs stop execution instead of being silently repaired.

**Interpretation:** The expected cohort is 23,151 snapshots, 12,447 patients, and 1,345 positive snapshots. MONTHLY has 12 positions; QUARTERLY has 4. These names describe enrollment buckets only: no historical feature reconstruction or averaging/summing of monthly feature values occurs. Patient separation prevents the same patient crossing partitions; it cannot remove upstream encoding limitations.

In [ ]:
def exact_dates(values, name):
    require(not values.isna().any(), name + ': missing dates are not allowed.')
    require(not values.map(lambda v: isinstance(v, (bool, np.bool_, int, float, np.number))).any(),
            name + ': numeric/Boolean values cannot be interpreted as dates.')
    dates = pd.to_datetime(values, errors='raise')
    require(dates.dt.tz is None and dates.eq(dates.dt.normalize()).all(), name + ': expected exact dates without timezone/time of day.')
    return dates.dt.strftime('%Y-%m-%d')


def snapshot_records(metadata):
    return [[r.PATIENT_ID, r.END_DT, int(r.RESP)] for r in metadata.itertuples()]


def require_prepared_selection(manifest=None):
    """Consume only the explicitly reviewed output of the clean preparation notebook."""
    require(isinstance(EXPECTED_PREPARED_MANIFEST_SHA256, str)
            and re.fullmatch(r'[0-9a-f]{64}', EXPECTED_PREPARED_MANIFEST_SHA256) is not None,
            'Set EXPECTED_PREPARED_MANIFEST_SHA256 to the reviewed notebook-01 manifest hash in 02/03/04.')
    if manifest is not None:
        require(digest_json(manifest) == EXPECTED_PREPARED_MANIFEST_SHA256,
                'Prepared manifest differs from the reviewed output. Do not automatically change the pin.')


def load_prepared():
    require_prepared_selection()
    blobs = read_artifacts(PREPARED_TABLE, PREPARED_NAMES)
    manifest = json.loads(blobs['manifest.json'])
    require_prepared_selection(manifest)
    require(manifest.get('schema') == 5 and manifest.get('dataset_id') == DATASET_ID,
            'Prepared dataset ID/schema changed.')
    require(manifest.get('padding_contract') == 'independent_timestep_availability_v1'
            and manifest.get('feature_policy') == 'v63_snapshot_only_valid_timestep_broadcast_v1', 'Unexpected preparation contract.')
    require(manifest.get('source_vintage') == '20260825', 'Unexpected source vintage.')
    require(isinstance(manifest.get('implementation_sha256'), str)
            and re.fullmatch(r'[0-9a-f]{64}', manifest['implementation_sha256']) is not None,
            'Missing preparation implementation fingerprint.')
    require(manifest.get('orientation') == '0=newest; snapshot-relative month buckets; newest ends at END_DT',
            'Prepared timestep orientation changed.')
    require(manifest.get('bucket_definition') == {
        'MONTHLY': {'steps': 12, 'months_per_step': 1},
        'QUARTERLY': {'steps': 4, 'months_per_step': 3},
        'anchor': 'snapshot calendar month',
        'start': 'first day of anchor month minus width*step minus (width-1) months',
        'end': 'min(END_DT, last day of anchor month minus width*step months)'},
        'Bucket definition changed; quarterly buckets must align with monthly triples.')
    features = selected_features(manifest)
    require(manifest.get('preprocessing_policy') == 'one_vector_per_eligible_train_snapshot_shared_v1', 'Unexpected preprocessing policy.')
    require(isinstance(manifest.get('frozen_split_assignments_sha256'), str)
            and re.fullmatch(r'[0-9a-f]{64}', manifest['frozen_split_assignments_sha256']) is not None,
            'Missing frozen split assignment fingerprint.')
    require(manifest.get('input_description') == 'V63 snapshot features repeated only at enrollment-valid timesteps; no historical feature reconstruction',
            'Input description must identify the snapshot-only repeated values.')
    _, _, encoding = load_business_parameters(features)
    require(encoding == manifest.get('business_encoding'), 'Frozen V63 encoding changed after preparation.')
    population = json.loads(blobs['population.json'])
    metadata = pd.DataFrame(population, columns=['PATIENT_ID', 'END_DT', 'RESP'])
    metadata = normalize_metadata(metadata).sort_values(['PATIENT_ID', 'END_DT']).reset_index(drop=True)
    require(population == snapshot_records(metadata), 'Prepared snapshot order changed.')
    population_check(metadata)
    source = normalize_metadata(read_table(PREFIX + '_SNAPSHOTS').select('PATIENT_ID', 'END_DT', 'RESP').toPandas())
    source = source.sort_values(['PATIENT_ID', 'END_DT']).reset_index(drop=True)
    require(metadata.equals(source), 'Frozen population/RESP changed after preparation.')
    require(digest_json(population) == manifest.get('population_sha256'), 'Population hash mismatch.')
    require(set(manifest.get('representations', {})) == {'MONTHLY', 'QUARTERLY'}, 'Both representations required.')
    bundles = {}
    for name, count in (('MONTHLY', 12), ('QUARTERLY', 4)):
        with np.load(io.BytesIO(blobs[name + '.npz']), allow_pickle=False) as arrays:
            require(set(arrays.files) == {'X', 'valid'}, 'Unexpected prepared array names.')
            X, valid = arrays['X'].copy(), arrays['valid'].copy()
        require(X.dtype == np.float32 and X.shape == (len(metadata), count, len(features))
                and valid.shape == X.shape[:2] and valid.dtype == bool, 'Invalid tensor or mask dimensions/type.')
        require(not np.isinf(X).any() and np.isfinite(X[~valid]).all() and np.all(X[~valid] == 0),
                'Invalid padding/infinite input; individual feature NaNs belong only to valid timesteps.')
        info = manifest['representations'][name]
        coverage = info.get('coverage_audit', {})
        require(coverage.get('unknown_timesteps') == 0, 'Unknown coverage must be resolved before preparation.')
        require(coverage.get('available_timesteps') == int(valid.sum())
                and coverage.get('unavailable_timesteps') == int((~valid).sum())
                and coverage.get('all_padded_snapshots') == int((~valid.any(axis=1)).sum()),
                'Coverage audit disagrees with the saved mask.')
        require(isinstance(coverage.get('policy'), str) and bool(coverage['policy'].strip())
                and isinstance(coverage.get('evidence'), str) and bool(coverage['evidence'].strip()),
                'Independent source-coverage evidence and policy are required.')
        require(digest_json(coverage) == info.get('availability_sha256'), 'Coverage audit fingerprint changed.')
        require(isinstance(info.get('provenance'), list) and len(info['provenance']) > 0,
                'Missing source/builder provenance.')
        require(array_hash(X) == info.get('X_sha256') and array_hash(valid) == info.get('valid_sha256'),
                'Tensor/mask fingerprint changed.')
        require(info.get('shape') == list(X.shape) and info.get('value_space') == 'V63 encoded',
                'Unexpected tensor shape/value space; do not re-encode upstream values.')
        for feature_index, row in enumerate(manifest['audit']):
            if row['VALUE_SOURCE'] == 'V63_SNAPSHOT':
                first_valid = valid.argmax(axis=1)
                reference = X[np.arange(len(X)), first_valid, feature_index][:, None]
                repeated = np.broadcast_to(reference, valid.shape)
                require(np.array_equal(X[:, :, feature_index][valid], repeated[valid], equal_nan=True),
                        row['FEATURE_NAME'] + ': snapshot fallback varies within a snapshot.')
        bundles[name] = {'raw_X': X, 'valid': valid, 'representation': name}
    monthly, quarterly = bundles['MONTHLY'], bundles['QUARTERLY']
    require(np.array_equal(quarterly['valid'], monthly['valid'].reshape(len(metadata), 4, 3).any(axis=2)),
            'Quarterly validity must be OR over the three corresponding monthly positions.')
    both = monthly['valid'].any(axis=1) & quarterly['valid'].any(axis=1)
    rows = np.flatnonzero(both)
    for feature_index, row in enumerate(manifest['audit']):
        if row['VALUE_SOURCE'] == 'V63_SNAPSHOT':
            a = monthly['raw_X'][rows, monthly['valid'][rows].argmax(axis=1), feature_index]
            b = quarterly['raw_X'][rows, quarterly['valid'][rows].argmax(axis=1), feature_index]
            require(np.array_equal(a, b, equal_nan=True), 'Snapshot fallback differs between representations.')
    return metadata, features, bundles, manifest


def split_statistics(metadata):
    out = []
    sets = {name: set(metadata.loc[metadata.SPLIT.eq(name), 'PATIENT_ID']) for name in ('train', 'validation', 'test')}
    expected = {'train': (16256, 8712, 941), 'validation': (3481, 1867, 202), 'test': (3414, 1868, 202)}
    for name, ids in sets.items():
        part = metadata.loc[metadata.SPLIT.eq(name)]
        require((len(part), len(ids), int(part.RESP.sum())) == expected[name], 'Original split counts changed: ' + name)
        out.append({'SPLIT': name, 'PATIENTS': len(ids), 'SNAPSHOTS': len(part), 'RESP_0': int(part.RESP.eq(0).sum()),
                    'RESP_1': int(part.RESP.sum()), 'POSITIVE_RATE': float(part.RESP.mean())})
    for a, b in (('train', 'validation'), ('train', 'test'), ('validation', 'test')):
        require(not sets[a].intersection(sets[b]), 'Patient overlap: ' + a + '/' + b)
    return pd.DataFrame(out)


def normalize_metadata(frame):
    require(not frame.columns.duplicated().any(), 'Duplicate metadata columns.')
    out = frame[['PATIENT_ID', 'END_DT', 'RESP']].copy().reset_index(drop=True)
    require(not out.empty and not out.isna().any().any(), 'Missing snapshot keys or labels.')
    require(out.PATIENT_ID.map(lambda x: isinstance(x, str) and bool(x) and x == x.strip()).all(),
            'Patient IDs must remain exact nonempty strings without surrounding whitespace.')
    out['END_DT'] = exact_dates(out.END_DT, 'END_DT')
    require(not out.RESP.map(lambda x: isinstance(x, (str, bool, np.bool_))).any()
            and out.RESP.isin([0, 1]).all(), 'RESP must be numeric 0 or 1, without coercion.')
    out['RESP'] = out.RESP.astype('int64')
    require(not out.duplicated(['PATIENT_ID', 'END_DT']).any(), 'Duplicate patient/date keys; no automatic deduplication.')
    return out


def bind_split(metadata, frozen):
    original = normalize_metadata(metadata).sort_values(['PATIENT_ID', 'END_DT']).reset_index(drop=True)
    normalized = normalize_metadata(frozen)
    normalized['SPLIT'] = frozen.SPLIT.to_numpy()
    normalized = normalized.sort_values(['PATIENT_ID', 'END_DT']).reset_index(drop=True)
    require(original.equals(normalized[['PATIENT_ID', 'END_DT', 'RESP']]),
            'Frozen split differs from the prepared snapshots/labels; exact set equality is required.')
    require(not normalized.SPLIT.isna().any() and set(normalized.SPLIT) == {'train', 'validation', 'test'},
            'Unexpected or missing frozen split labels.')
    require(not normalized.groupby('PATIENT_ID').SPLIT.nunique().gt(1).any(), 'Patient leakage between splits.')
    records = [[r.PATIENT_ID, r.END_DT, int(r.RESP), r.SPLIT] for r in normalized.itertuples()]
    hashes = {'snapshot_manifest_sha256': digest_json(records),
              'assignment_contract': 'exact_frozen_patient_split_v1'}
    split_statistics(normalized)
    return normalized, hashes


def population_check(metadata):
    observed = (len(metadata), metadata.PATIENT_ID.nunique(), int(metadata.RESP.sum()))
    if observed != (23151, 12447, 1345):
        raise ValueError(f"Frozen V63 cohort changed: snapshots/patients/positives = {observed}")


### One-value-per-snapshot TRAIN preprocessing

**Why:** Define median imputation and standardization using one feature vector per eligible TRAIN snapshot, taken from its first valid position. Share the resulting frozen statistics across MONTHLY and QUARTERLY, apply them to other partitions, and leave padded positions exactly zero.

**Expected output:** None while functions are defined. Later calls fail if a feature has no finite eligible TRAIN observation or if the shared saved statistics cannot be reproduced.

**Interpretation:** A snapshot contributes once, regardless of the number of eligible positions, so longer coverage no longer increases its preprocessing weight. Patients with more snapshots can still contribute more; this is snapshot weighting, not equal patient weighting. Padding is excluded from fitting. Missing feature values within eligible positions are imputed rather than making the whole timestep invalid.

In [ ]:
def fit_temporal_preprocessor(X, valid, rows):
    """One vector per eligible TRAIN snapshot; repeated covered periods do not change fit weight."""
    require(X.ndim == 3 and valid.shape == X.shape[:2] and valid.dtype == bool, 'Invalid input/mask alignment.')
    require(not np.isinf(X).any() and np.isfinite(X[~valid]).all() and np.all(X[~valid] == 0),
            'Invalid input or nonzero/nonfinite padding.')
    rows = np.asarray(rows)
    require(rows.ndim == 1 and np.issubdtype(rows.dtype, np.integer) and len(rows) > 0
            and len(np.unique(rows)) == len(rows) and (rows >= 0).all() and (rows < len(X)).all(),
            'TRAIN rows must be unique valid snapshot indices.')
    eligible = rows[valid[rows].any(axis=1)]
    require(len(eligible) > 0, 'No eligible TRAIN snapshots for preprocessing.')
    first = valid[eligible].argmax(axis=1)
    observed = X[eligible, first, :]
    repeated = np.broadcast_to(observed[:, None, :], X[eligible].shape)
    require(np.array_equal(X[eligible][valid[eligible]], repeated[valid[eligible]], equal_nan=True),
            'Snapshot-only feature values must be constant across valid timesteps.')
    require(not np.isnan(observed).all(axis=0).any(),
            'A feature is missing for every eligible TRAIN snapshot; resolve source values before modeling.')
    return fit_preprocessor(observed)


def transform_temporal(X, valid, state):
    require(X.ndim == 3 and valid.shape == X.shape[:2] and valid.dtype == bool, 'Invalid input/mask alignment.')
    require(not np.isinf(X).any() and np.isfinite(X[~valid]).all(), 'Invalid input or nonfinite padding.')
    # Do not standardize placeholders; transform only genuinely available timesteps.
    values = np.zeros(X.shape, dtype=np.float32)
    if valid.any():
        values[valid], _ = transform_features(X[valid], state)
    require(np.isfinite(values).all(), 'Nonfinite Transformer input.')
    return values


def load_experiment():
    metadata, features, bundles, manifest = load_prepared()
    blobs = read_artifacts(SPLIT_TABLE, SPLIT_NAMES)
    audit = json.loads(blobs['audit.json'])
    states = json.loads(blobs['preprocessing.json'])
    frozen = read_table(PREFIX + '_PATIENT_SPLIT').select('PATIENT_ID', 'END_DT', 'RESP', 'SPLIT').toPandas()
    metadata, hashes = bind_split(metadata, frozen)
    split_statistics(metadata)
    require(hashes['snapshot_manifest_sha256'] == manifest['frozen_split_assignments_sha256'],
            'Frozen split assignments changed since preparation.')
    require(hashes == audit['source_split_hashes'] and digest_json(manifest) == audit['manifest_sha256'],
            'Saved split audit changed.')
    records = [[r.PATIENT_ID, r.END_DT, int(r.RESP), r.SPLIT] for r in metadata.itertuples()]
    require(records == json.loads(blobs['split.json']), 'Saved assignments changed.')
    require(digest_json(states) == audit['preprocessing_sha256'], 'Saved preprocessing changed.')
    require(audit.get('preprocessing_policy') == 'one_vector_per_eligible_train_snapshot_shared_v1',
            'This snapshot-only experiment requires once-per-snapshot shared preprocessing.')
    require(set(states) == {'MONTHLY', 'QUARTERLY'} and states['MONTHLY'] == states['QUARTERLY'],
            'Monthly and quarterly must share identical preprocessing state.')
    indices = {s: np.flatnonzero(metadata.SPLIT.eq(s).to_numpy()) for s in ('train', 'validation', 'test')}
    experiments = {}
    for name, b in bundles.items():
        state = states[name]
        require(state == fit_temporal_preprocessor(b['raw_X'], b['valid'], indices['train']),
                'Preprocessing does not reproduce the once-per-eligible-TRAIN-snapshot fit.')
        X = transform_temporal(b['raw_X'], b['valid'], state)
        experiments[name] = dict(b, X=X, y=metadata.RESP.to_numpy(dtype=np.float32), metadata=metadata,
            indices=indices, features=features, preprocessor=state,
            hashes={'manifest_sha256': digest_json(manifest), 'snapshot_manifest_sha256': hashes['snapshot_manifest_sha256'],
                    'preprocessing_sha256': digest_json(state), 'model_input_sha256': array_hash(X),
                    'mask_sha256': array_hash(b['valid']),
                    'preprocessing_policy': 'one_vector_per_eligible_train_snapshot_shared_v1'})
    require(np.array_equal(experiments['MONTHLY']['y'], experiments['QUARTERLY']['y']), 'Representation labels differ.')
    return experiments, manifest, audit


def fit_preprocessor(X_train):
    if X_train.ndim != 2 or not len(X_train) or np.isinf(X_train).any():
        raise ValueError("Invalid training feature matrix.")
    all_missing = np.isnan(X_train).all(axis=0)
    median = np.array([0.0 if missing else np.nanmedian(X_train[:, i])
                       for i, missing in enumerate(all_missing)])
    filled = np.where(np.isnan(X_train), median, X_train)
    mean = filled.mean(axis=0)
    scale = filled.std(axis=0)
    scale[scale == 0] = 1.0
    if not np.isfinite(np.r_[median, mean, scale]).all():
        raise ValueError("Nonfinite preprocessing statistics.")
    return {"median": median.tolist(), "mean": mean.tolist(), "scale": scale.tolist(),
            "all_missing_train": all_missing.tolist()}


def transform_features(X, state):
    if X.ndim != 2 or X.shape[1] != len(state["median"]) or np.isinf(X).any():
        raise ValueError("Feature shape or values changed.")
    mask = np.isnan(X)
    values = ((np.where(mask, state["median"], X) - state["mean"]) / state["scale"]).astype(np.float32)
    if not np.isfinite(values).all():
        raise ValueError("Nonfinite standardized values.")
    return values, mask.astype(np.float32)


### Load and inspect the approved 49-feature input

**Why:** Execute the prepared-input checks and expose the feature-source audit before any preprocessing is fitted.

**Expected output:** Feature count 49, an explicit snapshot-only input description, and one audit row per feature with `VALUE_SOURCE=V63_SNAPSHOT`.

**Interpretation:** This confirms which predictors will enter the model and that all come from the matching V63 snapshot. A source label is provenance, not importance. Every eligible position carries that snapshot's same values; additional eligible months do not supply additional clinical observations. If a TEMPORAL source appears, the artifact belongs to a different design and must not be silently consumed here.

In [ ]:
# Load verified monthly and quarterly inputs without repeating V63 encoding.
metadata = features = bundles = manifest = None
metadata, features, bundles, manifest = load_prepared()
print('Confirmed feature count:', len(features), '; existing V63 encoding retained.')



print('Input design:', manifest['input_description'])
display(pd.DataFrame(manifest['audit'])[['FEATURE_NAME', 'VALUE_SOURCE']])


### Reuse and verify the frozen patient partitions

**Why:** Bind both representations to exactly the same pre-existing patient assignment, so their performance comparison is not driven by different cohorts.

**Expected output:** TRAIN 16,256 snapshots / 8,712 patients / 941 positives; VALIDATION 3,481 / 1,867 / 202; TEST 3,414 / 1,868 / 202. All pairwise patient overlaps must be zero and assignments must match across representations.

**Interpretation:** Positive rates are the random-ranking reference for average precision and lift. Material rate differences can affect metric comparisons even with no patient leakage. These are checks of the declared frozen partition, not a newly randomized split or evidence that previously examined TEST results become unseen again.

In [ ]:
# Bind both representations to the original patient split
frozen = read_table(PREFIX + '_PATIENT_SPLIT').select('PATIENT_ID', 'END_DT', 'RESP', 'SPLIT').toPandas()
metadata, source_split_hashes = bind_split(metadata, frozen)
require(source_split_hashes['snapshot_manifest_sha256'] == manifest['frozen_split_assignments_sha256'],
        'Frozen split assignments changed since preparation.')
statistics = split_statistics(metadata)
display(statistics)
display(pd.DataFrame([{'TRAIN_VALIDATION_OVERLAP': 0, 'TRAIN_TEST_OVERLAP': 0, 'VALIDATION_TEST_OVERLAP': 0,
                       'MONTHLY_QUARTERLY_ASSIGNMENTS_IDENTICAL': True}]))
indices = {s: np.flatnonzero(metadata.SPLIT.eq(s).to_numpy()) for s in ('train', 'validation', 'test')}
print('Same source keys, labels and frozen source patient assignments verified; no new splits created.')


### Fit and audit shared snapshot-level TRAIN preprocessing

**Why:** Fit the shared preprocessing state once per eligible TRAIN snapshot, then confirm both representations use identical statistics and produce finite inputs without altering padding.

**Expected output:** Preprocessing and coverage audit rows, including eligible TRAIN snapshots/positions, imputation counts, all-padded snapshots, and confirmation that transformed inputs are finite and the preprocessing state is shared.

**Interpretation:** Tensor-entry imputation counts can include repeated copies and all partitions; only one vector per eligible TRAIN snapshot contributes fitted statistics. Zero imputed entries does not prove historical availability. All-padded snapshots stay in the cohort but supply no fitting vector. Different valid-position counts describe coverage, not different preprocessing weights or independent sample sizes. Shared scaling isolates later comparison from a representation-specific preprocessing difference.

In [ ]:
# Fit once per eligible TRAIN snapshot and share the identical state across representations.
shared_state = fit_temporal_preprocessor(bundles['MONTHLY']['raw_X'], bundles['MONTHLY']['valid'], indices['train'])
quarterly_check = fit_temporal_preprocessor(bundles['QUARTERLY']['raw_X'], bundles['QUARTERLY']['valid'], indices['train'])
require(shared_state == quarterly_check, 'Shared snapshot values/eligibility must produce identical preprocessing.')
preprocessing, preprocessing_coverage = {}, []
for name, b in bundles.items():
    preprocessing[name] = dict(shared_state)
    values = transform_temporal(b['raw_X'], b['valid'], shared_state)
    require(values.shape == b['raw_X'].shape and np.all(values[~b['valid']] == 0), 'Transformed padding changed.')
    preprocessing_coverage.append({'MODEL': name,
        'ELIGIBLE_TRAIN_SNAPSHOTS_FITTED_ONCE': int(b['valid'][indices['train']].any(axis=1).sum()),
        'VALID_TRAIN_TIMESTEPS': int(b['valid'][indices['train']].sum()),
        'IMPUTED_FEATURE_VALUES_ALL_SPLITS': int(np.isnan(b['raw_X'][b['valid']]).sum()),
        'ALL_PADDED_SNAPSHOTS': int((~b['valid'].any(axis=1)).sum()),
        'ALL_MODEL_VALUES_FINITE': bool(np.isfinite(values).all()),
        'SHARED_PREPROCESSING': True})
display(pd.DataFrame(preprocessing_coverage))


### Save and reproduce the split and preprocessing contract

**Why:** Save patient assignments, fitted preprocessing, and their audit so training and evaluation consume the same immutable inputs. Reload them immediately and verify source assignments, hashes, and TRAIN-only fitted statistics.

**Expected output:** Successful private-table write/read-back confirmation, then the snapshot-only input description and 49-feature source audit.

**Interpretation:** A successful result means notebook 03 can reproduce this preparation. It is not a training result or a performance claim. If data, mask, or feature contracts change, produce a reviewed new prepared artifact and propagate its hash; do not bypass a mismatch by changing a fingerprint without checking the underlying change.

In [ ]:
# Save frozen preprocessing and the identical split manifest
split_rows = [[r.PATIENT_ID, r.END_DT, int(r.RESP), r.SPLIT] for r in metadata.itertuples()]
split_audit = {'manifest_sha256': digest_json(manifest), 'source_split_hashes': source_split_hashes,
               'preprocessing_sha256': digest_json(preprocessing), 'split_summary': statistics.to_dict('records'),
               'fit_scope': 'One vector per eligible TRAIN snapshot; identical shared state for monthly and quarterly',
               'preprocessing_policy': 'one_vector_per_eligible_train_snapshot_shared_v1',
               'new_assignments_created': False, 'monthly_quarterly_assignments_identical': True}
save_artifacts(SPLIT_TABLE, {'split.json': canonical_json(split_rows).encode(),
    'preprocessing.json': canonical_json(preprocessing).encode(), 'audit.json': canonical_json(split_audit).encode()})
experiments, _, _ = load_experiment()
print('Read-back verified. Continue with notebook 03.')

print('Input design:', manifest['input_description'])
display(pd.DataFrame(manifest['audit'])[['FEATURE_NAME', 'VALUE_SOURCE']])
